# Analisis Exploratorio de Datos (EDA)

En este archivo se lleva acabo un analisis exloratorio sobre el dataset de personas desaparecidas ... TODO

**Integrantes:**  
1. José Rubén Alfaro González  
2. Daniela Sanluis
3. Edwin Rodriguez 

**Profesor:**  
Jessica Santizo Galicia

**Ayudantes:**  
1. Diego Antonio Villalba González 
2. Ares Gael Castro Romero

**Número de tarea:**  
[Tarea #02]

# Configuración e importación de librerias
Configuramos el ambiente para trabajar. Usaremos librerias como pandas para la lectura de datos

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime
import warnings
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

print("Librerias importadas exitosamente")

# Inicio del análisis

## 1. Carga del dataset
Cargamos el dataset. Además, podemos ver la forma del DS, que contiene 133,886 filas, y 11 columnas. Más información respecto a la información que proporciona cada una de las columnas será dada en el pdf que adjuntamos. 

In [ ]:
# Cargar dataset
df = pd.read_csv('../data/data_secretariado.csv')
f = df.shape[0]
col = df.shape[1]

print(f"Dataset cargado exitosamente")
print("Número de filas:", f)
print("Número de columnas:", col)

print("Primera inspección: ")
df.head()

También podemos ver los tipos de datos con los que vamos a contar. La mayoría serán datos categoricos o temporales, pero a partir de estos podemos calcular ciertos datos que nos serán de utilidad como la edad al momento de la desaparición (lo haremos más adelante).

In [ ]:
df.info()

## 2. Análisis de datos faltantes y confidenciales
### Cantidad de datos faltantes y confidenciales
A continuación se muestra la cantidad de datos nulos del dataset. Podemos ver que la mayoría de datos nulos son de los campos de fechas, ya sea de nacimiento, desaparicion, o de registro. Además, es un porcentaje bastante alto de información perdida, llegando casi a un 60% para fechas de nacimiento, y 43% para las demás fechas. Fuera de eso, las demás columnas no tienen datos faltantes, pero falta revisar la cantidad de datos confidenciales.

También se muestra la cantidad de información confidencial. Se puede ver que la cantidad de información confidencial en cada columna es la misma, por lo que podemos intuir que son casos que siguen siendo confidenciales debido a varias razones (casos abiertos, en proceso de investigación, etc.)

In [ ]:
missing = pd.DataFrame({
    'Columna': df.columns,
    'Valores_Nulos': df.isnull().sum(),
    'Porcentaje': (df.isnull().sum() / len(df)) * 100
}).sort_values('Valores_Nulos', ascending=False)

print("Valores nulos por columna:")
print(missing.to_string(index=False))

confidential = pd.DataFrame({
    'Columna': df.columns,
    'Valores_Confidenciales': [(df[col] == 'CONFIDENCIAL').sum() for col in df.columns],
    'Porcentaje': [(df[col] == 'CONFIDENCIAL').sum() / len(df) * 100 for col in df.columns]
}).sort_values('Valores_Confidenciales', ascending=False)

print("\nValores confidenciales por columna:")
print(confidential.to_string(index=False))


In [ ]:
def faltantes_por_entidad(df, columna, contar_confidencial=True):
    resultados = []
    for entidad, sub_df in df.groupby("ENTIDAD"):
        registros = len(sub_df)
        nulos = sub_df[columna].isnull().sum()
        confidencial = (sub_df[columna] == "CONFIDENCIAL").sum() if contar_confidencial else 0
        faltantes = nulos + confidencial
        porcentaje = (faltantes / registros) * 100
        resultados.append({
            "Entidad": entidad.upper(),
            "Porcentaje": round(porcentaje, 2)
        })
    return pd.DataFrame(resultados)

Además, con un poco (bastante) de ayuda de Claude, podemos visualizar de manera más efectiva la cantidad de información faltante por cada uno de los estados y el porcentaje correspondiente. Lo que si vemos, es que la mayoría de la información perdida no es debido a que no se registre, sino que hay una gran cantidad de información confidencial. Además, no parece haber relación alguna entre datos faltantes y los estados, sin embargo, si es posible ver que hay estados como Jalisco que tienen mucha información faltante en la mayoría de las columnas. 

In [ ]:
# Cargar GeoJSON
mexico = gpd.read_file("https://raw.githubusercontent.com/angelnmara/geojson/master/mexicoHigh.json")
mexico["name"] = mexico["name"].str.upper()

# Mapeo de nombres si es necesario
mapeo = {
    "COAHUILA DE ZARAGOZA": "COAHUILA",
    "VERACRUZ DE IGNACIO DE LA LLAVE": "VERACRUZ",
    "MICHOACÁN DE OCAMPO": "MICHOACÁN",
    # agregar los que no coincidan
    "ESTADO DE MÉXICO": "MÉXICO",
}

def mostrar_mapa(columna, contar_confidencial):
    datos = faltantes_por_entidad(df, columna, contar_confidencial)
    datos["Entidad"] = datos["Entidad"].replace(mapeo)
    
    # Desglose por entidad
    desglose = []
    for entidad, sub_df in df.groupby("ENTIDAD"):
        registros = len(sub_df)
        nulos = sub_df[columna].isnull().sum()
        confidencial = (sub_df[columna] == "CONFIDENCIAL").sum()
        desglose.append({
            "Entidad": entidad.upper(),
            "Registros": registros,
            "Nulos": nulos,
            "Confidencial": confidencial,
            "Total_Faltantes": nulos + confidencial,
            "% Faltantes": round((nulos + confidencial) / registros * 100, 2)
        })
    desglose = pd.DataFrame(desglose).sort_values("% Faltantes", ascending=False)
    print(f"\nDesglose de datos faltantes en: {columna}")
    print(desglose.to_string(index=False))
    
    # Mapa
    mapa = mexico.merge(datos, left_on="name", right_on="Entidad", how="left")
    
    fig, ax = plt.subplots(1, 1, figsize=(15, 10))
    mapa.plot(
        column="Porcentaje",
        cmap="YlOrRd",
        linewidth=0.8,
        edgecolor="black",
        legend=True,
        legend_kwds={"label": "% Datos Faltantes", "shrink": 0.6},
        missing_kwds={"color": "lightgrey", "label": "Sin datos"},
        vmin=0,
        vmax=100,
        ax=ax
    )
    titulo = f"Datos faltantes en {columna} por entidad"
    titulo += " (incluye CONFIDENCIAL)" if contar_confidencial else " (solo nulos)"
    ax.set_title(titulo, fontsize=14)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

columnas = [col for col in df.columns if col not in ["ID_VICTIMA", "ENTIDAD", "CVE_ENT"]]

widgets.interact(
    mostrar_mapa,
    columna=columnas,
    contar_confidencial=widgets.Checkbox(value=True, description="Incluir CONFIDENCIAL")
)

## 3. Analisis de localización
Las variables que estaremos utilizando serán las siguientes:

EDAD_DESAPARICION: Sacaremos moda, media, mediana y cuartiles

SEXO: Sacaremos moda

ESTATUS_VICTIMA: Sacaremos moda

ENTIDAD: Sacaremos moda

MUNICIPIO: Sacaremos moda

FECHA_DESAPARICION: Sacaremos moda y mediana

En la mayoría de las variables anteriores, la información nula es mínima, sin embargo, la cantidad de información confidencial es considerable, y podría meter ruido al análisis de los datos. Además, en algunos estados la perdida de información llega incluso a 60% y en casos como los de Campeche que 60% de 135 nos deja con solamente ~50 casos con información útil, lo cual hace imposible hacer una estadistica más general para el estado. Aún así, la información confidencial no aporta más allá de lo visualizado anteriormente, y no podemos "rellenar" los datos pues no hay ni (1) suficiente información y (2) los casos confidenciales no tienen nada de información que nos permita "intuir" las demás medidad pues toda su información es confidencial. Por eso, decidimos crear un sub-dataset con la información confidencial transformada como NaN.

Ahora, también hay una probabilidad de que los casos confidenciales sean todos de un tipo en particular, o que representen a cierto grupo de los datos en particular, por lo que no tener esa información podría introducir cierto sesgo en el análisis.

In [ ]:
df_clean = df.copy()
date_columns = ['FECHA_NACIMIENTO', 'FECHA_DESAPARICION', 'FECHA_REGISTRO', 'SEXO', 'ESTATUS_VICTIMA', 'MUNICIPIO']
for col in date_columns:
    df_clean[col] = df_clean[col].replace('CONFIDENCIAL', pd.NA)

df_clean.head()


df_clean['FECHA_NACIMIENTO'] = pd.to_datetime(df_clean['FECHA_NACIMIENTO'], errors='coerce')
df_clean['FECHA_DESAPARICION'] = pd.to_datetime(df_clean['FECHA_DESAPARICION'], errors='coerce')
df_clean['FECHA_REGISTRO'] = pd.to_datetime(df_clean['FECHA_REGISTRO'], errors='coerce')

'''
Además, agregamos una nueva columna 'EDAD_DESAPARICION' que calcula la edad de la víctima al momento de su desaparición,
restando la fecha de nacimiento de la fecha de desaparición y dividiendo por 365 para obtener la edad en años.
'''
df_clean['EDAD_DESAPARICION'] = (df_clean['FECHA_DESAPARICION'] - df_clean['FECHA_NACIMIENTO']).dt.days // 365

df_clean.head()

### Paso 2
Obtenemos un poco de información respecto al dataset. Calculamos la media, moda, mediana, cantidad de valores no-nulos, etc. de los valores de las columnas, así como el tipo de valores que usan (las columnas que lo permiten).

In [ ]:
def get_stats(series):
    """
    Obtiene la moda, media (si aplica) y mediana de la columna proporcionada

    Args:
        series: Serie de pandas a analizar
        column: La columna a la que se le desea obtener información

    Returns:
        Media (si es numérica), mediana y moda de la serie
    """
    mode = series.mode().iloc[0] if not series.mode().empty else pd.NA
    median = series.median() if pd.api.types.is_numeric_dtype(series) else pd.NA
    mean = series.mean() if pd.api.types.is_numeric_dtype(series) else pd.NA
    return mean, median, mode

print("Moda de las columnas: ")
mean, median , mode = get_stats(df_clean)
print(mode)

print("\nOtra información del dataset: ")
df_clean.describe(include='all')

### Medidas de localización:
EDAD_DESAPARICION: Sacaremos moda, media, mediana y cuartiles

SEXO: Sacaremos moda

ESTATUS_VICTIMA: Sacaremos moda

ENTIDAD: Sacaremos moda

MUNICIPIO: Sacaremos moda

FECHA_DESAPARICION: Sacaremos moda y mediana


### Medidas de variabilidad:
EDAD_DESAPARICION: Sacaremos varianza, rango, IQR? y desviación estandar

FECHA_DESAPARICION: Sacaremos el minimo y máximo, para así obtener el intervalo temporal

FECHA_REGISTRO: Igualmente sacaremos el rango

### Medidas de heterogeneidad
SEXO: Sacaremos la entropía de Shannon, y el índice de Simpson

ESTATUS_VICTIMA: Sacaremos la entropía de Shannon

ENTIDAD: Sacaremos la entropía de Shannon

MUNICIPIO: Sacaremos la entropía de Shannon

ORIGEN_REPORTE: Sacaremos la entropía de Shannon


### Medidas de concentración
SEXO: Sacaremos la frecuencia relativa a la moda

ENTIDAD: Sacaremos el HHI

MUNICIPIO: Sacaremos el HHI

EDAD_DESAPARICION: Sacaremos el índice de Gini y la curva de Lorenz